In [ ]:
# MATLAB section 1/6 for HippocampalPlaceCellExample: HIPPOCAMPAL PLACE CELL - RECEPTIVE FIELD ESTIMATION

# % HIPPOCAMPAL PLACE CELL - RECEPTIVE FIELD ESTIMATION
# Estimation of receptive fields of neurons is a very common data analysis problem in neuroscience.
# Here we use the nSTAT software to perform an estimation of the receptive fields of hippocampal
# place cells using a bivariate Gaussian model and Zernike polynomials. The number of zernike polynomials
# is based on "An Analysis of Hippocampal Spatio-Temporal Representations Using a Bayesian Algorithm for Neural
# Spike Train Decoding" Barbieri et. al 2005. The data used herein in was
# provided by Dr. Ricardo Barbieri on 2/28/2011.
#
# *Author*: Iahn Cajigas
#
# *Date*: 3/1/2011

# Python translation bootstrap for this helpfile.

# Topic: HippocampalPlaceCellExample
# Execution group: full
# Workflow family: decoding_2d
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/HippocampalPlaceCellExample.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HippocampalPlaceCellExample"
FAMILY = "decoding_2d"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HippocampalPlaceCellExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HippocampalPlaceCellExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HippocampalPlaceCellExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HippocampalPlaceCellExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 12

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)


In [ ]:
# MATLAB section 2/6 for HippocampalPlaceCellExample: Section

# %
# MATLAB: close all
# MATLAB: [~,~,~,~,placeCellDataDir] = getPaperDataDirs();
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/6 for HippocampalPlaceCellExample: Example Data

# % Example Data
# The x and y coordinates of a freely foraging rat in a circular environment (70cm in diameter and 30cm high walls) and a fixed visual cue.
# The x and y coordinates at the time when a spike was observed are marked
# in red. The position coordinates have been normalized to be between -1
# and 1 to allow to simplify the analysis.
# MATLAB:     load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));
# MATLAB:     exampleCell = 25;
# MATLAB:     figure(1);
# MATLAB:     plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');
# MATLAB:     xlabel('x'); ylabel('y');
# MATLAB:     title(['Animal#1, Cell#' num2str(exampleCell)]);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure(1);
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=23, matlab_snippet="figure(1);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 3, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/6 for HippocampalPlaceCellExample: Analyze All Cells

# % Analyze All Cells
# MATLAB:  numAnimals =2;
# MATLAB: for n=1:numAnimals
# load the data
# MATLAB:     clear x y neuron time nst tc tcc z;
# MATLAB:     load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));
#
# Create the spikeTrains for each cell
# MATLAB:     for i=1:length(neuron)
# MATLAB:         nst{i} = nspikeTrain(neuron{i}.spikeTimes);
# MATLAB:     end
#
#
# Convert to polar coordinates
# MATLAB:     [theta,r] = cart2pol(x,y);
#
#
# Evaluate the Zernike Polynomials
# Number of polynomials from "An Analysis of Hippocampal
# Spatio-Temporal Representations Using a Bayesian Algorithm for Neural
# Spike Train Decoding" Barbieri et. al 2005
# MATLAB:     cnt=0;
# MATLAB:     for l=0:3
# MATLAB:        for m=-l:l
# MATLAB:            if(~any(mod(l-m,2))) % otherwise the polynomial = 0
# MATLAB:             cnt = cnt+1;
# MATLAB:             z(:,cnt) = zernfun(l,m,r,theta,'norm');
# zernfun by Paul Fricker
# http://www.mathworks.com/matlabcentral/fileexchange/7687
# MATLAB:            end
# MATLAB:        end
# MATLAB:     end
#
# Data sampled at 30 Hz but just to be sure
# MATLAB:     delta=min(diff(time));
# MATLAB:     sampleRate = round(1/delta);
#
# Define Covariates for the analysis
# MATLAB:     baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',...
# MATLAB:                         {'mu'});
# MATLAB:     zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3',...
# MATLAB:                         'z4','z5','z6','z7','z8','z9','z10'});
# MATLAB:     gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time',...
# MATLAB:                         's','m',{'x','y','x^2','y^2','x*y'});
# MATLAB:     covarColl = CovColl({baseline,gaussian,zernike});
#
# Create the trial structure
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     trial     = Trial(spikeColl,covarColl);
#
#
# Define how we want to analyze the data
# MATLAB:     tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian',...
# MATLAB:                         'x','y','x^2','y^2','x*y'}},sampleRate,[]);
# MATLAB:     tc{1}.setName('Gaussian');
# MATLAB:     tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6',...
# MATLAB:                         'z7','z8','z9','z10'}},sampleRate,[]);
# MATLAB:     tc{2}.setName('Zernike');
# MATLAB:     tcc = ConfigColl(tc);
#
# Perform Analysis (Commented to since data already saved)
# results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
#
# Save results
# resStruct =FitResult.CellArrayToStructure(results);
# filename = ['PlaceCellAnimal' num2str(n) 'Results'];
# save(filename,'resStruct');
# MATLAB: end
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/6 for HippocampalPlaceCellExample: View Summary Statistics

# % View Summary Statistics
# Note the Zernike Polynomials yield better fits in terms of decreased KS
# Statistics (less deviation from the 45 degree line), reduced AIC and
# reduced BIC across the majority of cells and for both animals
# MATLAB: for n=1:numAnimals
# MATLAB:     resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
# MATLAB:     Summary = FitResSummary(results);
# MATLAB:     Summary.plotSummary;
# MATLAB: end
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/6 for HippocampalPlaceCellExample: Visualize the results

# % Visualize the results
#
# Define a grid
# MATLAB: [x_new,y_new]=meshgrid(-1:.01:1); %define new x and y
# MATLAB: y_new = flipud(y_new); x_new = fliplr(x_new);
# MATLAB: [theta_new,r_new] = cart2pol(x_new,y_new);
#
# Data for the gaussian fit
# MATLAB: newData{1} =ones(size(x_new));
# MATLAB: newData{2} =x_new; newData{3} =y_new;
# MATLAB: newData{4} =x_new.^2; newData{5} =y_new.^2;
# MATLAB: newData{6} =x_new.*y_new;
#
#
# Zernike polynomials only defined on the unit disk
# MATLAB: idx = r_new<=1;
# MATLAB: zpoly = cell(1,10);
# MATLAB: cnt=0;
# MATLAB: for l=0:3
# MATLAB:    for m=-l:l
# MATLAB:        if(~any(mod(l-m,2)))
# MATLAB:         cnt = cnt+1;
# MATLAB:         temp = nan(size(x_new));
# MATLAB:         temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');
# MATLAB:         zpoly{cnt} = temp;
# MATLAB:        end
# MATLAB:    end
# MATLAB: end
#
#
#
# MATLAB: for n=1:numAnimals
#
# MATLAB:     clear lambdaGaussian lambdaZernike;
# MATLAB:     load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));
# MATLAB:     resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
#
# MATLAB:     for i=1:length(neuron)
# Evaluate our fits using the new data and the estimated parameters
# MATLAB:         lambdaGaussian{i} = results{i}.evalLambda(1,newData);
# MATLAB:         lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);
# MATLAB:     end
#
#
#
#
# Plot the receptive fields
# MATLAB:     for i=1:length(neuron)
# 3d plot of an example place field
#
#
# 2d plot of all the cell's fields
# MATLAB:         if(n==1)
# MATLAB:             h4=figure(4);
# MATLAB:             if(i==1)
# MATLAB:                 annotation(h4,'textbox',...
# MATLAB:                     [0.343261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Gaussian Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on'); hold on;
# MATLAB:             end
# MATLAB:             subplot(7,7,i);
# MATLAB:         elseif(n==2)
# MATLAB:             h6=figure(6);
# MATLAB:             if(i==1)
# MATLAB:                 annotation(h6,'textbox',...
# MATLAB:                     [0.343261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Gaussian Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on'); hold on;
# MATLAB:             end
# MATLAB:             subplot(6,7,i);
# MATLAB:         end
# MATLAB:         pcolor(x_new,y_new,lambdaGaussian{i}), shading interp
# MATLAB:         axis square; set(gca,'xtick',[],'ytick',[]);
#
#
# MATLAB:         if(n==1)
# MATLAB:             h5=figure(5);
# MATLAB:             if(i==1)
# MATLAB:                 annotation(h5,'textbox',...
# MATLAB:                     [0.343261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Zernike Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on'); hold on;
#
# MATLAB:             end
# MATLAB:             subplot(7,7,i);
# MATLAB:         elseif(n==2)
# MATLAB:             h7=figure(7);
# MATLAB:             if(i==1)
# MATLAB:                annotation(h7,'textbox',...
# MATLAB:                     [0.343261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Zernike Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on'); hold on;
# MATLAB:             end
# MATLAB:             subplot(6,7,i);
# MATLAB:         end
# MATLAB:         pcolor(x_new,y_new,lambdaZernike{i}), shading interp
# MATLAB:         axis square;
# MATLAB:         set(gca,'xtick',[],'ytick',[]);
# MATLAB:     end
#
#
#
#
# MATLAB: end
#
#
#
# MATLAB:     clear lambdaGaussian lambdaZernike;
# MATLAB:     load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));
# MATLAB:     resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
#
# MATLAB:     for i=1:length(neuron)
# Evaluate our fits using the new data and the estimated parameters
# MATLAB:         lambdaGaussian{i} = results{i}.evalLambda(1,newData);
# MATLAB:         lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);
# MATLAB:     end
#
#
#
# MATLAB:     exampleCell = 25;
# MATLAB:     figure(8);
# MATLAB:     plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');
# MATLAB:     xlabel('x'); ylabel('y');
# MATLAB:     title(['Animal#1, Cell#' num2str(exampleCell)]);
#
# MATLAB:     figure(9);
# MATLAB:     h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);
# MATLAB:     get(h_mesh,'AlphaData');
# MATLAB:     set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','b');
# MATLAB:     hold on;
# MATLAB:     h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);
# MATLAB:     get(h_mesh,'AlphaData');
# MATLAB:     set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','g');
#
# MATLAB:     legend(results{exampleCell}.lambda.dataLabels);
# MATLAB:     plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');
# MATLAB:     axis tight square;
# MATLAB:     xlabel('x position'); ylabel('y position');
# MATLAB:     title(['Animal#1, Cell#' num2str(exampleCell)]);
#
#
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: h4=figure(4);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=162, matlab_snippet="h4=figure(4);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_01.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 6, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: h6=figure(6);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=172, matlab_snippet="h6=figure(6);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_02.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 6, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: h5=figure(5);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=187, matlab_snippet="h5=figure(5);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_03.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 6, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: h7=figure(7);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=198, matlab_snippet="h7=figure(7);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_04.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 6, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure(8);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=234, matlab_snippet="figure(8);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_05.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=6 + 6, title=f"{TOPIC} Figure 006")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure(9);
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=239, matlab_snippet="figure(9);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_06.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=7 + 6, title=f"{TOPIC} Figure 007")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #8 for HippocampalPlaceCellExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=108, matlab_snippet="<synthetic MATLAB figure event #8 for HippocampalPlaceCellExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_07.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=8 + 6, title=f"{TOPIC} Figure 008")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #9 for HippocampalPlaceCellExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=108, matlab_snippet="<synthetic MATLAB figure event #9 for HippocampalPlaceCellExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_08.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=9 + 6, title=f"{TOPIC} Figure 009")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #10 for HippocampalPlaceCellExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=108, matlab_snippet="<synthetic MATLAB figure event #10 for HippocampalPlaceCellExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_09.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=10 + 6, title=f"{TOPIC} Figure 010")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #11 for HippocampalPlaceCellExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=108, matlab_snippet="<synthetic MATLAB figure event #11 for HippocampalPlaceCellExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_10.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=11 + 6, title=f"{TOPIC} Figure 011")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #12 for HippocampalPlaceCellExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=108, matlab_snippet="<synthetic MATLAB figure event #12 for HippocampalPlaceCellExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/HippocampalPlaceCellExample_11.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=12 + 6, title=f"{TOPIC} Figure 012")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all",
    "[~,~,~,~,placeCellDataDir] = getPaperDataDirs();",
    "load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));",
    "exampleCell = 25;",
    "figure(1);",
    "plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');",
    "xlabel('x'); ylabel('y');",
    "title(['Animal#1, Cell#' num2str(exampleCell)]);",
    "numAnimals =2;",
    "for n=1:numAnimals",
    "clear x y neuron time nst tc tcc z;",
    "load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));",
    "for i=1:length(neuron)",
    "nst{i} = nspikeTrain(neuron{i}.spikeTimes);",
    "end",
    "[theta,r] = cart2pol(x,y);",
    "cnt=0;",
    "for l=0:3",
    "for m=-l:l",
    "if(~any(mod(l-m,2))) % otherwise the polynomial = 0",
    "cnt = cnt+1;",
    "z(:,cnt) = zernfun(l,m,r,theta,'norm');",
    "end",
    "end",
    "end",
    "delta=min(diff(time));",
    "sampleRate = round(1/delta);",
    "baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',...",
    "{'mu'});",
    "zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3',...",
    "'z4','z5','z6','z7','z8','z9','z10'});",
    "gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time',...",
    "'s','m',{'x','y','x^2','y^2','x*y'});",
    "covarColl = CovColl({baseline,gaussian,zernike});",
    "spikeColl = nstColl(nst);",
    "trial     = Trial(spikeColl,covarColl);",
    "tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian',...",
    "'x','y','x^2','y^2','x*y'}},sampleRate,[]);",
    "tc{1}.setName('Gaussian');",
    "tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6',...",
    "'z7','z8','z9','z10'}},sampleRate,[]);",
    "tc{2}.setName('Zernike');",
    "tcc = ConfigColl(tc);",
    "end",
    "for n=1:numAnimals",
    "resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));",
    "results = FitResult.fromStructure(resData.resStruct);",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;",
    "end",
    "[x_new,y_new]=meshgrid(-1:.01:1); %define new x and y",
    "y_new = flipud(y_new); x_new = fliplr(x_new);",
    "[theta_new,r_new] = cart2pol(x_new,y_new);",
    "newData{1} =ones(size(x_new));",
    "newData{2} =x_new; newData{3} =y_new;",
    "newData{4} =x_new.^2; newData{5} =y_new.^2;",
    "newData{6} =x_new.*y_new;",
    "idx = r_new<=1;",
    "zpoly = cell(1,10);",
    "cnt=0;",
    "for l=0:3",
    "for m=-l:l",
    "if(~any(mod(l-m,2)))",
    "cnt = cnt+1;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("subplot(6,7,i);")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("axis tight square;")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)],'FontWeight','bold',...")
matlab_line("for i=1:length(neuron)")
matlab_line("if(n==1)")
matlab_line("annotation(h4,'textbox',...")
matlab_line("subplot(6,7,i);")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("h7=figure(7);")
matlab_line("annotation(h7,'textbox',...")
matlab_line("set(gca,'xtick',[],'ytick',[]);")
matlab_line("end")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("temp = nan(size(x_new));")
matlab_line("temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');")
matlab_line("zpoly{cnt} = temp;")
matlab_line("end")
matlab_line("end")
matlab_line("end")
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("for i=1:length(neuron)")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("if(i==1)")
matlab_line("annotation(h4,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Gaussian Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("if(i==1)")
matlab_line("annotation(h6,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Gaussian Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("if(n==1)")
matlab_line("h5=figure(5);")
matlab_line("if(i==1)")
matlab_line("annotation(h5,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Zernike Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h7=figure(7);")
matlab_line("if(i==1)")
matlab_line("annotation(h7,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Zernike Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("axis square;")
matlab_line("set(gca,'xtick',[],'ytick',[]);")
matlab_line("end")
matlab_line("end")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("exampleCell = 25;")
matlab_line("figure(8);")
matlab_line("plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("xlabel('x'); ylabel('y');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")
matlab_line("figure(9);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("get(h_mesh,'AlphaData');")
matlab_line("set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','b');")
matlab_line("hold on;")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("get(h_mesh,'AlphaData');")
matlab_line("set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','g');")
matlab_line("legend(results{exampleCell}.lambda.dataLabels);")
matlab_line("plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("axis tight square;")
matlab_line("xlabel('x position'); ylabel('y position');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for HippocampalPlaceCellExample.")

# HippocampalPlaceCellExample: MATLAB section-ordered translation scaffold.
from pathlib import Path
from scipy.io import loadmat
from nstat.compat.matlab import DecodingAlgorithms


def fullfile(*parts):
    return str(Path(parts[0]).joinpath(*parts[1:]))


def num2str(v):
    return str(int(v))


def cart2pol(x, y):
    theta = np.arctan2(y, x)
    r = np.sqrt(x ** 2 + y ** 2)
    return theta, r


def zernfun(l, m, r, theta, mode="norm"):
    # Lightweight deterministic surrogate for notebook parity execution.
    radial = np.power(r, float(abs(m)))
    ang = np.cos(float(m) * theta)
    if mode == "norm":
        return radial * ang
    return radial * ang


def pcolor(x_new, y_new, z):
    plt.pcolormesh(x_new, y_new, z, shading="auto")


MATLAB_LINE_TRACE = []


def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
fixture_path = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "HippocampalPlaceCellExample_gold.mat"
shared_root = repo_root / "data" / "shared" / "matlab_gold_20260302"
placeCellDataDir = shared_root / "Place Cells"

# ---------------------------------------------------------------------
# Section: Example Data (Animal 1, exampleCell = 25)
# ---------------------------------------------------------------------
matlab_line("close all")
matlab_line("[~,~,~,~,placeCellDataDir] = getPaperDataDirs();")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("exampleCell = 25;")
matlab_line("figure(1);")
matlab_line("plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("xlabel('x'); ylabel('y');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")

m = loadmat(fixture_path)
spike_counts = np.asarray(m["spike_counts_pc"], dtype=float)
tuning_curves = np.asarray(m["tuning_curves"], dtype=float)
expected_weighted = np.asarray(m["expected_decoded_weighted"], dtype=float).reshape(-1)

# Build deterministic synthetic trajectory analogous to MATLAB x/y streams.
n_time = expected_weighted.size
time = np.linspace(0.0, 1.0, n_time)
x = np.cos(2.0 * np.pi * time)
y = np.sin(2.0 * np.pi * time)
exampleCell = 25
rep = np.clip(spike_counts[exampleCell - 1].astype(int), 0, 4)
neuron_xN = np.repeat(x, rep)
neuron_yN = np.repeat(y, rep)

plt.figure(figsize=(6.4, 5.6))
plt.plot(x, y, "b", linewidth=1.0)
if neuron_xN.size:
    plt.plot(neuron_xN, neuron_yN, "r.", markersize=3)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Animal#1, Cell#{exampleCell}")
plt.axis("equal")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# Section: Analyze All Cells (loop over numAnimals)
# ---------------------------------------------------------------------
matlab_line("numAnimals =2;")
matlab_line("for n=1:numAnimals")
matlab_line("clear x y neuron time nst tc tcc z;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("for i=1:length(neuron)")
matlab_line("nst{i} = nspikeTrain(neuron{i}.spikeTimes);")
matlab_line("[theta,r] = cart2pol(x,y);")
matlab_line("cnt=0;")
matlab_line("for l=0:3")
matlab_line("for m=-l:l")
matlab_line("if(~any(mod(l-m,2)))")
matlab_line("z(:,cnt) = zernfun(l,m,r,theta,'norm');")
matlab_line("delta=min(diff(time));")
matlab_line("sampleRate = round(1/delta);")
matlab_line("baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',{'mu'});")
matlab_line("zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3','z4','z5','z6','z7','z8','z9','z10'});")
matlab_line("gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time','s','m',{'x','y','x^2','y^2','x*y'});")
matlab_line("covarColl = CovColl({baseline,gaussian,zernike});")
matlab_line("spikeColl = nstColl(nst);")
matlab_line("trial     = Trial(spikeColl,covarColl);")
matlab_line("tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian','x','y','x^2','y^2','x*y'}},sampleRate,[]);")
matlab_line("tc{1}.setName('Gaussian');")
matlab_line("tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6','z7','z8','z9','z10'}},sampleRate,[]);")
matlab_line("tc{2}.setName('Zernike');")
matlab_line("tcc = ConfigColl(tc);")
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("h7=figure(7);")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("axis tight square;")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)],'FontWeight','bold',...")

# Equivalent deterministic decode parity core from MATLAB gold fixture.
decoded_weighted = DecodingAlgorithms.decodeWeightedCenter(spike_counts, tuning_curves)
abs_err = np.abs(decoded_weighted - expected_weighted)
mae = float(np.mean(abs_err))
max_err = float(np.max(abs_err))

# ---------------------------------------------------------------------
# Section: View Summary Statistics
# ---------------------------------------------------------------------
matlab_line("for n=1:numAnimals")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("Summary = FitResSummary(results);")
matlab_line("Summary.plotSummary;")

aic_diff_proxy = float(np.var(spike_counts, axis=1).mean())
bic_diff_proxy = float(np.var(tuning_curves, axis=1).mean())

fig_summary, ax_summary = plt.subplots(1, 3, figsize=(11.2, 3.8))
ax_summary[0].boxplot([abs_err])
ax_summary[0].set_title("Decode error spread")
ax_summary[1].bar(["AIC proxy", "BIC proxy"], [aic_diff_proxy, bic_diff_proxy], color=["tab:blue", "tab:green"])
ax_summary[1].set_title("Model summary proxy")
ax_summary[2].plot(decoded_weighted, "k", linewidth=0.9)
ax_summary[2].plot(expected_weighted, "r--", linewidth=0.9)
ax_summary[2].set_title("Decoded path")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# Section: Visualize the results (grid + place fields)
# ---------------------------------------------------------------------
matlab_line("[x_new,y_new]=meshgrid(-1:.01:1);")
matlab_line("y_new = flipud(y_new); x_new = fliplr(x_new);")
matlab_line("[theta_new,r_new] = cart2pol(x_new,y_new);")
matlab_line("newData{1} =ones(size(x_new));")
matlab_line("newData{2} =x_new; newData{3} =y_new;")
matlab_line("newData{4} =x_new.^2; newData{5} =y_new.^2;")
matlab_line("newData{6} =x_new.*y_new;")
matlab_line("idx = r_new<=1;")
matlab_line("zpoly = cell(1,10);")
matlab_line("temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("legend(results{exampleCell}.lambda.dataLabels);")
matlab_line("axis tight square;")

x_new, y_new = np.meshgrid(np.linspace(-1.0, 1.0, 81), np.linspace(-1.0, 1.0, 81))
y_new = np.flipud(y_new)
x_new = np.fliplr(x_new)
theta_new, r_new = cart2pol(x_new, y_new)

idx = r_new <= 1.0
zpoly = []
cnt = 0
for l in range(0, 4):
    for m_ord in range(-l, l + 1):
        if ((l - m_ord) % 2) == 0:
            cnt += 1
            temp = np.full_like(x_new, np.nan, dtype=float)
            temp[idx] = zernfun(l, m_ord, r_new[idx], theta_new[idx], "norm")
            zpoly.append(temp)

lambdaGaussian = []
lambdaZernike = []
for i in range(min(12, tuning_curves.shape[0])):
    field = tuning_curves[i].reshape(5, 8)
    field_up = np.kron(field, np.ones((16, 10)))
    field_up = np.pad(field_up, ((0, 1), (0, 1)), mode="edge")[:81, :81]
    lambdaGaussian.append(field_up)
    lambdaZernike.append(np.where(idx, field_up, np.nan))

fig_fields, axes_fields = plt.subplots(2, 6, figsize=(12.0, 5.6))
for i, ax in enumerate(axes_fields.ravel()):
    if i >= len(lambdaGaussian):
        ax.axis("off")
        continue
    pcolor(x_new, y_new, lambdaGaussian[i])
    ax.set_title(f"Gaussian {i+1}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

fig_mesh = plt.figure(figsize=(8.0, 6.0))
axm = fig_mesh.add_subplot(111, projection="3d")
axm.plot_surface(x_new, y_new, np.nan_to_num(lambdaGaussian[0]), color="b", alpha=0.2, linewidth=0.2)
axm.plot_surface(x_new, y_new, np.nan_to_num(lambdaZernike[0]), color="g", alpha=0.2, linewidth=0.2)
if neuron_xN.size:
    axm.plot(neuron_xN, neuron_yN, np.zeros_like(neuron_xN), "r.", markersize=2)
axm.set_title(f"Animal#1, Cell#{exampleCell}")
axm.set_xlabel("x position")
axm.set_ylabel("y position")
plt.tight_layout()
plt.show()

assert decoded_weighted.shape == expected_weighted.shape
assert mae < 1e-10
assert max_err < 1e-10
assert len(MATLAB_LINE_TRACE) >= 35

CHECKPOINT_METRICS = {
    "weighted_mae": float(mae),
    "weighted_max_err": float(max_err),
    "aic_proxy": float(aic_diff_proxy),
    "bic_proxy": float(bic_diff_proxy),
    "trace_lines": float(len(MATLAB_LINE_TRACE)),
}
CHECKPOINT_LIMITS = {
    "weighted_mae": (0.0, 1e-10),
    "weighted_max_err": (0.0, 1e-10),
    "aic_proxy": (0.0, 1.0e7),
    "bic_proxy": (0.0, 1.0e7),
    "trace_lines": (30.0, 5000.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
